# 🗺️ Graph Representation — Runnable Notebook

Companion to [`README.md`](README.md) and
[`06_graph_representation.html`](06_graph_representation.html).

The same graph, two ways: **adjacency matrix** vs **adjacency list** — then a head-to-head comparison.

## 1. Adjacency matrix
A `V x V` grid, `1` = edge. `O(1)` edge check, but `O(V^2)` space.

In [ ]:
class MatrixGraph:
    """Adjacency MATRIX: V x V grid. O(V^2) space, O(1) edge check."""
    def __init__(self, n, directed=False):
        self.n = n
        self.directed = directed
        self.m = [[0] * n for _ in range(n)]

    def add_edge(self, u, v):
        self.m[u][v] = 1
        if not self.directed:
            self.m[v][u] = 1              # undirected -> symmetric grid

    def has_edge(self, u, v):
        return self.m[u][v] == 1          # O(1) — the matrix's superpower

    def neighbours(self, u):
        return [v for v in range(self.n) if self.m[u][v]]   # O(V): scan the row

print("MatrixGraph defined")

## 2. Adjacency list
Per vertex, just its neighbours. `O(V+E)` space, `O(degree)` ops.

In [ ]:
from collections import defaultdict

class ListGraph:
    """Adjacency LIST: vertex -> neighbours. O(V+E) space, O(degree) ops."""
    def __init__(self, directed=False):
        self.directed = directed
        self.adj = defaultdict(list)

    def add_edge(self, u, v):
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)

    def has_edge(self, u, v):
        return v in self.adj[u]           # O(degree(u)): scan u's list

    def neighbours(self, u):
        return self.adj[u]                # O(degree(u))

print("ListGraph defined")

## 3. Build the same graph both ways — and check they agree

In [ ]:
edges = [(0,1), (0,3), (1,2), (1,4), (2,5), (3,4), (4,5)]   # A..F as 0..5
mg = MatrixGraph(6)
lg = ListGraph()
for u, v in edges:
    mg.add_edge(u, v)
    lg.add_edge(u, v)

for u, v in [(0,1), (2,5), (0,2), (3,5)]:
    same = mg.has_edge(u, v) == lg.has_edge(u, v)
    print(f"edge {u}-{v}? matrix={mg.has_edge(u,v)}  list={lg.has_edge(u,v)}  agree={same}")
    assert same

print("neighbours of 1: matrix", mg.neighbours(1), "| list", sorted(lg.neighbours(1)))
assert mg.neighbours(1) == sorted(lg.neighbours(1))

## 4. Space, made concrete
The matrix is mostly zeros — that waste is why **sparse** graphs prefer the list.

In [ ]:
matrix_cells = mg.n * mg.n                              # every cell exists, edge or not
matrix_ones  = sum(sum(row) for row in mg.m)            # actual 1s
list_entries = sum(len(v) for v in lg.adj.values())     # only real edges (x2, undirected)

print(f"matrix: {matrix_cells} cells, but only {matrix_ones} are 1  ({matrix_cells - matrix_ones} wasted)")
print(f"list  : {list_entries} entries total   ->  O(V + E)")
assert matrix_cells == 36 and matrix_ones == 14 and list_entries == 14

## 5. The edge list (a third option)
Just the edges — great for edge-centric algorithms (Kruskal, Union-Find).

In [ ]:
def to_edge_list(lg):
    """Flat list of edges, each undirected edge counted once."""
    seen, out = set(), []
    for u in lg.adj:
        for v in lg.adj[u]:
            key = tuple(sorted((u, v)))
            if key not in seen:
                seen.add(key)
                out.append(key)
    return sorted(out)

print("edge list:", to_edge_list(lg))
assert len(to_edge_list(lg)) == 7

## ✅ Recap
| Operation | Matrix | List |
|---|---|---|
| Space | `O(V²)` | `O(V+E)` |
| Check edge u–v? | **O(1)** | `O(degree)` |
| List neighbours | `O(V)` | **O(degree)** |
| Best when… | **dense** | **sparse** (most real graphs) |

Undirected ⇒ the matrix is **symmetric**. Default to the **list** for real graphs.

Next: [`07_Graph_Traversal`](../07_Graph_Traversal/README.md).